# 08. M1 pre-event feature 확장 threshold 0.6 검증

## tl;dr

라벨과 window는 고정하고, feature만 `base70`에서 `expanded154`로 늘렸을 때 threshold 0.6 기준 성능이 좋아지는지 검증한다. 이번 단계는 최종 모델 선택이 아니라 feature 확장 효과 검증이다.

## Context & Methods

- 목표: `normal` vs `efd_possible` pre-event 분류
- window: 기존 `report_pre_7d` 유지
- dataset:
  - `strict_no_event20`: normal 35 + positive 14
  - `strict_no_event20_no_event67`: normal 35 + positive 13
- 제외:
  - Event 20: low coverage
  - Event 34: fault label unknown, audit 전용
  - Event 69: fault label unknown 및 training end 없음, audit 전용
- 모델:
  - `DummyClassifier`
  - `LogisticRegression(class_weight="balanced")`
- 비교:
  - `base70`: 기존 10개 센서 x 7개 통계
  - `expanded154`: base70 + derived signal 통계 + temporal delta feature
- 의사결정 기준: threshold 0.6

In [1]:
from __future__ import annotations

from pathlib import Path
import math
import warnings
import zipfile

import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=UserWarning)

ROOT = Path.cwd()
DATA_DIR = ROOT / "05_데이터셋" / "PreDist"
ZIP_PATH = DATA_DIR / "predist_dataset.zip"
OUTPUT_DIR = ROOT / "07_데이터산출물"

AUDIT_PATH = OUTPUT_DIR / "m1_feature_expansion_audit.csv"
FEATURES_PATH = OUTPUT_DIR / "m1_feature_expansion_features.csv"
DATASET_SUMMARY_PATH = OUTPUT_DIR / "m1_feature_expansion_dataset_summary.csv"
CV_METRICS_PATH = OUTPUT_DIR / "m1_feature_expansion_cv_metrics.csv"
CV_PREDICTIONS_PATH = OUTPUT_DIR / "m1_feature_expansion_cv_predictions.csv"
THRESHOLD_METRICS_PATH = OUTPUT_DIR / "m1_feature_expansion_threshold_metrics.csv"
FEATURE_IMPORTANCE_PATH = OUTPUT_DIR / "m1_feature_expansion_feature_importance.csv"
DECISION_MATRIX_PATH = OUTPUT_DIR / "m1_feature_expansion_decision_matrix.csv"
REPORT_PATH = OUTPUT_DIR / "08_M1_pre_event_feature_확장_threshold_검증_보고서.md"

SOURCE_PREFIX = "manufacturer 1"
RANDOM_STATE = 42
POSITIVE_LABEL = "efd_possible"
EVENT20_ID = 20
EVENT34_ID = 34
EVENT67_ID = 67
EVENT69_ID = 69
DECISION_THRESHOLD = 0.6
THRESHOLDS = [0.4, 0.5, 0.6, 0.7]

ORIGINAL_SIGNALS = [
    "outdoor_temperature",
    "s_hc1_supply_temperature",
    "s_hc1_supply_temperature_setpoint",
    "p_hc1_return_temperature",
    "p_net_meter_energy",
    "p_net_meter_volume",
    "p_net_meter_heat_power",
    "p_net_meter_flow",
    "p_net_supply_temperature",
    "p_net_return_temperature",
]
DERIVED_SIGNALS = [
    "s_hc1_supply_temperature_error",
    "p_net_delta_temperature",
    "p_net_power_flow_ratio",
    "p_return_gap",
]
ALL_SIGNALS = ORIGINAL_SIGNALS + DERIVED_SIGNALS
SENSOR_STATS = ["mean", "std", "min", "max", "median", "last_minus_first", "missing_rate"]
TEMPORAL_STATS = [
    "last_1d_mean_minus_prev_6d_mean",
    "last_12h_mean_minus_prev_12h_mean",
    "last_6h_mean_minus_prev_6h_mean",
    "last_1d_std_minus_prev_6d_std",
]

BASE_FEATURE_COLUMNS = [f"{signal}__{stat}" for signal in ORIGINAL_SIGNALS for stat in SENSOR_STATS]
DERIVED_STAT_FEATURE_COLUMNS = [f"{signal}__{stat}" for signal in DERIVED_SIGNALS for stat in SENSOR_STATS]
TEMPORAL_FEATURE_COLUMNS = [f"{signal}__{stat}" for signal in ALL_SIGNALS for stat in TEMPORAL_STATS]
EXPANDED_FEATURE_COLUMNS = BASE_FEATURE_COLUMNS + DERIVED_STAT_FEATURE_COLUMNS + TEMPORAL_FEATURE_COLUMNS
FEATURE_SET_COLUMNS = {
    "base70": BASE_FEATURE_COLUMNS,
    "expanded154": EXPANDED_FEATURE_COLUMNS,
}

LABEL_TO_BINARY = {"normal": 0, POSITIVE_LABEL: 1}
BINARY_TO_LABEL = {0: "normal", 1: POSITIVE_LABEL}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
assert ZIP_PATH.exists(), ZIP_PATH
assert len(BASE_FEATURE_COLUMNS) == 70
assert len(EXPANDED_FEATURE_COLUMNS) == 154

## Data

In [2]:
def read_metadata(name: str) -> pd.DataFrame:
    with zipfile.ZipFile(ZIP_PATH) as zf:
        with zf.open(f"{SOURCE_PREFIX}/{name}") as file:
            return pd.read_csv(file, sep=";")


def parse_datetime_columns(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    df = df.copy()
    for column in columns:
        if column in df.columns:
            df[column] = pd.to_datetime(df[column], errors="coerce")
    return df


faults = parse_datetime_columns(
    read_metadata("faults.csv"),
    ["Report date", "Possible anomaly start", "Possible anomaly end", "Training start", "Training end"],
)
normal_events = parse_datetime_columns(
    read_metadata("normal_events.csv"),
    ["Event start", "Event end", "Training start", "Training end"],
)
disturbances = parse_datetime_columns(read_metadata("disturbances.csv"), ["Event start"])

faults["Event ID"] = faults["Event ID"].astype(int)
faults["substation ID"] = faults["substation ID"].astype(int)
normal_events["Event ID"] = normal_events["Event ID"].astype(int)
normal_events["substation ID"] = normal_events["substation ID"].astype(int)
disturbances["substation ID"] = disturbances["substation ID"].astype(int)

strict_positive_ids = set(
    faults.loc[
        faults["efd_possible"].eq(True)
        & faults["Possible anomaly start"].notna()
        & faults["Possible anomaly end"].notna()
        & faults["Training start"].notna()
        & faults["Training end"].notna(),
        "Event ID",
    ].astype(int)
)
strict_positive_ids_no_event20 = sorted(strict_positive_ids - {EVENT20_ID})
strict_positive_ids_no_event20_event67 = sorted(strict_positive_ids - {EVENT20_ID, EVENT67_ID})

print("normal events:", len(normal_events))
print("strict positive ids:", sorted(strict_positive_ids))
print("strict positive ids without Event 20:", strict_positive_ids_no_event20)
print("strict positive ids without Event 20 and Event 67:", strict_positive_ids_no_event20_event67)

normal events: 35
strict positive ids: [1, 3, 7, 10, 15, 20, 40, 44, 47, 49, 52, 53, 57, 64, 67]
strict positive ids without Event 20: [1, 3, 7, 10, 15, 40, 44, 47, 49, 52, 53, 57, 64, 67]
strict positive ids without Event 20 and Event 67: [1, 3, 7, 10, 15, 40, 44, 47, 49, 52, 53, 57, 64]


## Feature Engineering

In [3]:
OPERATIONAL_CACHE: dict[int, pd.DataFrame] = {}


def load_operational(substation_id: int) -> pd.DataFrame:
    path = f"{SOURCE_PREFIX}/operational_data/substation_{int(substation_id)}.csv"
    with zipfile.ZipFile(ZIP_PATH) as zf:
        with zf.open(path) as file:
            df = pd.read_csv(file, sep=";")
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
    missing_columns = [column for column in ORIGINAL_SIGNALS if column not in df.columns]
    if missing_columns:
        raise ValueError(f"missing signal columns for substation {substation_id}: {missing_columns}")
    return df.sort_values("timestamp").reset_index(drop=True)


def get_operational(substation_id: int) -> pd.DataFrame:
    substation_id = int(substation_id)
    if substation_id not in OPERATIONAL_CACHE:
        OPERATIONAL_CACHE[substation_id] = load_operational(substation_id)
    return OPERATIONAL_CACHE[substation_id]


def add_derived_signals(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["s_hc1_supply_temperature_error"] = (
        pd.to_numeric(df["s_hc1_supply_temperature"], errors="coerce")
        - pd.to_numeric(df["s_hc1_supply_temperature_setpoint"], errors="coerce")
    )
    df["p_net_delta_temperature"] = (
        pd.to_numeric(df["p_net_supply_temperature"], errors="coerce")
        - pd.to_numeric(df["p_net_return_temperature"], errors="coerce")
    )
    flow = pd.to_numeric(df["p_net_meter_flow"], errors="coerce")
    heat_power = pd.to_numeric(df["p_net_meter_heat_power"], errors="coerce")
    df["p_net_power_flow_ratio"] = np.where(flow.abs() > 1e-9, heat_power / flow, np.nan)
    df["p_return_gap"] = (
        pd.to_numeric(df["p_hc1_return_temperature"], errors="coerce")
        - pd.to_numeric(df["p_net_return_temperature"], errors="coerce")
    )
    return df


def window_slice(substation_id: int, window_start: pd.Timestamp, window_end: pd.Timestamp) -> pd.DataFrame:
    df = get_operational(substation_id)
    mask = df["timestamp"].ge(window_start) & df["timestamp"].lt(window_end)
    return add_derived_signals(df.loc[mask, ["timestamp", *ORIGINAL_SIGNALS]].copy())


def expected_rows(window_start: pd.Timestamp, window_end: pd.Timestamp) -> int:
    minutes = (window_end - window_start).total_seconds() / 60
    return int(round(minutes / 10))


def disturbance_summary(substation_id: int, window_start: pd.Timestamp, window_end: pd.Timestamp) -> tuple[int, str]:
    mask = (
        disturbances["substation ID"].eq(int(substation_id))
        & disturbances["Event start"].ge(window_start)
        & disturbances["Event start"].lt(window_end)
    )
    window_disturbances = disturbances.loc[mask].copy()
    if window_disturbances.empty:
        return 0, ""
    disturbance_types = "|".join(sorted(window_disturbances["type"].dropna().astype(str).unique()))
    return int(len(window_disturbances)), disturbance_types


def safe_std(series: pd.Series) -> float:
    non_null = pd.to_numeric(series, errors="coerce").dropna()
    if len(non_null) > 1:
        return float(non_null.std(ddof=1))
    if len(non_null) == 1:
        return 0.0
    return np.nan


def stat_values(window_data: pd.DataFrame, signals: list[str]) -> dict[str, float]:
    values: dict[str, float] = {}
    for signal in signals:
        series = pd.to_numeric(window_data[signal], errors="coerce")
        non_null = series.dropna()
        values[f"{signal}__mean"] = float(non_null.mean()) if len(non_null) else np.nan
        values[f"{signal}__std"] = safe_std(series)
        values[f"{signal}__min"] = float(non_null.min()) if len(non_null) else np.nan
        values[f"{signal}__max"] = float(non_null.max()) if len(non_null) else np.nan
        values[f"{signal}__median"] = float(non_null.median()) if len(non_null) else np.nan
        values[f"{signal}__last_minus_first"] = (
            float(non_null.iloc[-1] - non_null.iloc[0])
            if len(non_null) > 1
            else 0.0
            if len(non_null) == 1
            else np.nan
        )
        values[f"{signal}__missing_rate"] = float(series.isna().mean()) if len(series) else 1.0
    return values


def segment_series(
    window_data: pd.DataFrame,
    signal: str,
    start: pd.Timestamp,
    end: pd.Timestamp,
) -> pd.Series:
    mask = window_data["timestamp"].ge(start) & window_data["timestamp"].lt(end)
    return pd.to_numeric(window_data.loc[mask, signal], errors="coerce")


def safe_mean(series: pd.Series) -> float:
    non_null = pd.to_numeric(series, errors="coerce").dropna()
    return float(non_null.mean()) if len(non_null) else np.nan


def temporal_values(
    window_data: pd.DataFrame,
    window_start: pd.Timestamp,
    window_end: pd.Timestamp,
    signals: list[str],
) -> dict[str, float]:
    values: dict[str, float] = {}
    for signal in signals:
        last_1d = segment_series(window_data, signal, window_end - pd.Timedelta(days=1), window_end)
        prev_6d = segment_series(window_data, signal, window_start, window_end - pd.Timedelta(days=1))
        last_12h = segment_series(window_data, signal, window_end - pd.Timedelta(hours=12), window_end)
        prev_12h = segment_series(
            window_data,
            signal,
            window_end - pd.Timedelta(hours=24),
            window_end - pd.Timedelta(hours=12),
        )
        last_6h = segment_series(window_data, signal, window_end - pd.Timedelta(hours=6), window_end)
        prev_6h = segment_series(
            window_data,
            signal,
            window_end - pd.Timedelta(hours=12),
            window_end - pd.Timedelta(hours=6),
        )
        values[f"{signal}__last_1d_mean_minus_prev_6d_mean"] = safe_mean(last_1d) - safe_mean(prev_6d)
        values[f"{signal}__last_12h_mean_minus_prev_12h_mean"] = safe_mean(last_12h) - safe_mean(prev_12h)
        values[f"{signal}__last_6h_mean_minus_prev_6h_mean"] = safe_mean(last_6h) - safe_mean(prev_6h)
        values[f"{signal}__last_1d_std_minus_prev_6d_std"] = safe_std(last_1d) - safe_std(prev_6d)
    return values


def make_sample(
    *,
    label: str,
    label_strength: str,
    source_event_id: int,
    substation_id: int,
    window_start: pd.Timestamp,
    window_end: pd.Timestamp,
    metadata: dict | None = None,
) -> dict:
    metadata = metadata or {}
    window_data = window_slice(substation_id, window_start, window_end)
    expected_sample_count = expected_rows(window_start, window_end)
    sample_count = int(len(window_data))
    coverage_rate = float(sample_count / expected_sample_count) if expected_sample_count else 0.0
    disturbance_count, disturbance_types = disturbance_summary(substation_id, window_start, window_end)

    row = {
        "sample_id": f"{label_strength}_{int(source_event_id):04d}",
        "manufacturer": "manufacturer_1",
        "label": label,
        "label_strength": label_strength,
        "source_event_id": int(source_event_id),
        "substation_id": int(substation_id),
        "window_start": window_start,
        "window_end": window_end,
        "window_days": float((window_end - window_start).total_seconds() / 86400),
        "window_policy": "normal_event_7d" if label == "normal" else "report_pre_7d",
        "sample_count": sample_count,
        "expected_sample_count": expected_sample_count,
        "coverage_rate": coverage_rate,
        "disturbance_count": disturbance_count,
        "disturbance_types": disturbance_types,
    }
    row.update(metadata)
    row.update(stat_values(window_data, ORIGINAL_SIGNALS))
    row.update(stat_values(window_data, DERIVED_SIGNALS))
    row.update(temporal_values(window_data, window_start, window_end, ALL_SIGNALS))
    return row

In [4]:
normal_rows = []
for _, event in normal_events.sort_values("Event ID").iterrows():
    normal_rows.append(
        make_sample(
            label="normal",
            label_strength="normal",
            source_event_id=int(event["Event ID"]),
            substation_id=int(event["substation ID"]),
            window_start=event["Event start"],
            window_end=event["Event end"],
            metadata={
                "source_type": "normal_event",
                "report_date": pd.NaT,
                "fault_label": "",
                "efd_possible": False,
                "event67_flag": False,
                "audit_status": "accepted",
                "exclusion_reason": "",
            },
        )
    )

positive_audit_rows = []
for _, event in faults.sort_values("Event ID").iterrows():
    event_id = int(event["Event ID"])
    if event_id not in strict_positive_ids and event_id not in {EVENT34_ID, EVENT69_ID}:
        continue

    fault_label = "" if pd.isna(event["Fault label"]) else str(event["Fault label"])
    fault_label_unknown = fault_label.strip().lower() == "unknown"
    report_date = event["Report date"]
    window_start = report_date - pd.Timedelta(days=7)
    window_end = report_date

    if event_id == EVENT20_ID:
        audit_status = "excluded"
        exclusion_reason = "event20_low_coverage"
    elif event_id == EVENT34_ID:
        audit_status = "audit_only"
        exclusion_reason = "fault_label_unknown"
    elif event_id == EVENT69_ID:
        audit_status = "audit_only"
        exclusion_reason = "fault_label_unknown_training_end_missing"
    else:
        audit_status = "accepted"
        exclusion_reason = ""

    row = make_sample(
        label=POSITIVE_LABEL,
        label_strength="strict_positive" if event_id in strict_positive_ids else "audit_only_unknown",
        source_event_id=event_id,
        substation_id=int(event["substation ID"]),
        window_start=window_start,
        window_end=window_end,
        metadata={
            "source_type": "fault_record",
            "report_date": report_date,
            "possible_anomaly_start": event["Possible anomaly start"],
            "possible_anomaly_end": event["Possible anomaly end"],
            "training_start": event["Training start"],
            "training_end": event["Training end"],
            "fault_label": fault_label,
            "fault_label_unknown": fault_label_unknown,
            "efd_possible": bool(event["efd_possible"]),
            "event67_flag": event_id == EVENT67_ID,
            "audit_status": audit_status,
            "exclusion_reason": exclusion_reason,
        },
    )
    positive_audit_rows.append(row)

audit_df = pd.DataFrame(positive_audit_rows)
accepted_positive_df = audit_df.loc[audit_df["audit_status"].eq("accepted")].copy()
feature_df = pd.concat([pd.DataFrame(normal_rows), accepted_positive_df], ignore_index=True)

audit_df.to_csv(AUDIT_PATH, index=False, encoding="utf-8-sig")
feature_df.to_csv(FEATURES_PATH, index=False, encoding="utf-8-sig")

print("audit rows:", len(audit_df))
print("feature rows:", len(feature_df))
print(audit_df[["source_event_id", "label_strength", "coverage_rate", "fault_label_unknown", "event67_flag", "audit_status", "exclusion_reason"]].to_string(index=False))

audit rows: 17
feature rows: 49
 source_event_id     label_strength  coverage_rate  fault_label_unknown  event67_flag audit_status                         exclusion_reason
               1    strict_positive       1.000000                False         False     accepted                                         
               3    strict_positive       1.000000                False         False     accepted                                         
               7    strict_positive       1.000000                False         False     accepted                                         
              10    strict_positive       1.000000                False         False     accepted                                         
              15    strict_positive       1.000000                False         False     accepted                                         
              20    strict_positive       0.724206                False         False     excluded                     event20_l

## Dataset Definition

In [5]:
def build_dataset(dataset_id: str) -> pd.DataFrame:
    normal_part = feature_df.loc[feature_df["label"].eq("normal")].copy()
    positive_part = feature_df.loc[feature_df["label"].eq(POSITIVE_LABEL)].copy()
    if dataset_id == "strict_no_event20":
        pass
    elif dataset_id == "strict_no_event20_no_event67":
        positive_part = positive_part.loc[~positive_part["source_event_id"].eq(EVENT67_ID)].copy()
    else:
        raise ValueError(dataset_id)
    dataset = pd.concat([normal_part, positive_part], ignore_index=True)
    dataset["dataset_id"] = dataset_id
    return dataset


DATASETS = {
    dataset_id: build_dataset(dataset_id)
    for dataset_id in ["strict_no_event20", "strict_no_event20_no_event67"]
}

summary_rows = []
for dataset_id, dataset in DATASETS.items():
    positive_events = sorted(dataset.loc[dataset["label"].eq(POSITIVE_LABEL), "source_event_id"].astype(int).tolist())
    summary_rows.append(
        {
            "dataset_id": dataset_id,
            "rows": int(len(dataset)),
            "normal_rows": int(dataset["label"].eq("normal").sum()),
            "positive_rows": int(dataset["label"].eq(POSITIVE_LABEL).sum()),
            "positive_event_ids": ",".join(map(str, positive_events)),
            "event20_included": EVENT20_ID in positive_events,
            "event34_included": EVENT34_ID in positive_events,
            "event67_included": EVENT67_ID in positive_events,
            "event69_included": EVENT69_ID in positive_events,
            "base_feature_count": len(BASE_FEATURE_COLUMNS),
            "expanded_feature_count": len(EXPANDED_FEATURE_COLUMNS),
            "coverage_min": float(dataset["coverage_rate"].min()),
            "coverage_median": float(dataset["coverage_rate"].median()),
        }
    )

dataset_summary_df = pd.DataFrame(summary_rows)
dataset_summary_df.to_csv(DATASET_SUMMARY_PATH, index=False, encoding="utf-8-sig")
dataset_summary_df

,dataset_id,rows,normal_rows,positive_rows,positive_event_ids,event20_included,event34_included,event67_included,event69_included,base_feature_count,expanded_feature_count,coverage_min,coverage_median
0,strict_no_event20,49,35,14,"1,3,7,10,15,40,44,47,49,52,53,57,64,67",False,False,True,False,70,154,0.99504,1.0
1,strict_no_event20_no_event67,48,35,13,"1,3,7,10,15,40,44,47,49,52,53,57,64",False,False,False,False,70,154,0.99504,1.0


## Training

In [6]:
def make_models() -> dict[str, Pipeline]:
    return {
        "dummy_most_frequent": Pipeline(
            [
                ("imputer", SimpleImputer(strategy="median")),
                ("model", DummyClassifier(strategy="most_frequent")),
            ]
        ),
        "logistic_balanced": Pipeline(
            [
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                (
                    "model",
                    LogisticRegression(
                        class_weight="balanced",
                        max_iter=2000,
                        solver="liblinear",
                        random_state=RANDOM_STATE,
                    ),
                ),
            ]
        ),
    }


def positive_scores(model: Pipeline, X: pd.DataFrame) -> np.ndarray:
    estimator = model.named_steps["model"]
    if hasattr(model, "predict_proba"):
        probabilities = model.predict_proba(X)
        classes = list(estimator.classes_)
        if 1 in classes:
            return probabilities[:, classes.index(1)]
    return np.zeros(len(X), dtype=float)


def metric_row(y_true: np.ndarray, y_pred: np.ndarray, y_score: np.ndarray | None) -> dict:
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    if y_score is not None and len(np.unique(y_true)) == 2:
        roc_auc = float(roc_auc_score(y_true, y_score))
    else:
        roc_auc = 0.5
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "roc_auc": roc_auc,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


def train_and_evaluate(dataset_id: str, feature_set: str, dataset: pd.DataFrame):
    dataset = dataset.copy().reset_index(drop=True)
    feature_columns = FEATURE_SET_COLUMNS[feature_set]
    X = dataset[feature_columns].copy()
    y = dataset["label"].map(LABEL_TO_BINARY).astype(int).to_numpy()
    groups = dataset["substation_id"].astype(str).to_numpy()

    splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    metric_rows = []
    prediction_rows = []
    importance_rows = []

    for fold, (train_index, test_index) in enumerate(splitter.split(X, y, groups), start=1):
        X_train, X_test = X.iloc[train_index], X.iloc[test_index]
        y_train, y_test = y[train_index], y[test_index]
        train_groups = sorted(set(groups[train_index]))
        test_groups = sorted(set(groups[test_index]))
        assert set(train_groups).isdisjoint(test_groups)
        assert len(np.unique(y_train)) == 2

        for model_name, model in make_models().items():
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test).astype(int)
            y_score = positive_scores(model, X_test)
            row = metric_row(y_test, y_pred, y_score)
            row.update(
                {
                    "dataset_id": dataset_id,
                    "feature_set": feature_set,
                    "model": model_name,
                    "fold": fold,
                    "feature_count": len(feature_columns),
                    "train_rows": int(len(train_index)),
                    "test_rows": int(len(test_index)),
                    "train_groups": "|".join(train_groups),
                    "test_groups": "|".join(test_groups),
                    "test_normal": int((y_test == 0).sum()),
                    "test_efd_possible": int((y_test == 1).sum()),
                }
            )
            metric_rows.append(row)

            prediction_meta = dataset.iloc[test_index][
                [
                    "sample_id",
                    "label",
                    "label_strength",
                    "source_event_id",
                    "substation_id",
                    "window_start",
                    "window_end",
                    "window_days",
                    "coverage_rate",
                    "sample_count",
                    "disturbance_count",
                    "event67_flag",
                ]
            ].copy()
            prediction_meta["dataset_id"] = dataset_id
            prediction_meta["feature_set"] = feature_set
            prediction_meta["fold"] = fold
            prediction_meta["model"] = model_name
            prediction_meta["actual_binary"] = y_test
            prediction_meta["predicted_binary"] = y_pred
            prediction_meta["predicted_label"] = [BINARY_TO_LABEL[int(value)] for value in y_pred]
            prediction_meta["positive_score"] = y_score
            prediction_meta["is_correct"] = prediction_meta["actual_binary"].eq(
                prediction_meta["predicted_binary"]
            )
            prediction_rows.extend(prediction_meta.to_dict("records"))

            if model_name == "logistic_balanced":
                coefficients = model.named_steps["model"].coef_[0]
                for feature, coefficient in zip(feature_columns, coefficients):
                    signal, stat = feature.split("__", 1)
                    importance_rows.append(
                        {
                            "dataset_id": dataset_id,
                            "feature_set": feature_set,
                            "fold": fold,
                            "feature": feature,
                            "signal": signal,
                            "stat": stat,
                            "coefficient": float(coefficient),
                            "abs_coefficient": float(abs(coefficient)),
                        }
                    )
    return metric_rows, prediction_rows, importance_rows


all_metric_rows = []
all_prediction_rows = []
all_importance_rows = []
for dataset_id, dataset in DATASETS.items():
    for feature_set in ["base70", "expanded154"]:
        metric_rows, prediction_rows, importance_rows = train_and_evaluate(dataset_id, feature_set, dataset)
        all_metric_rows.extend(metric_rows)
        all_prediction_rows.extend(prediction_rows)
        all_importance_rows.extend(importance_rows)

cv_metrics_df = pd.DataFrame(all_metric_rows)
cv_predictions_df = pd.DataFrame(all_prediction_rows)
feature_importance_raw_df = pd.DataFrame(all_importance_rows)

cv_metrics_df.to_csv(CV_METRICS_PATH, index=False, encoding="utf-8-sig")
cv_predictions_df.to_csv(CV_PREDICTIONS_PATH, index=False, encoding="utf-8-sig")

feature_importance_df = (
    feature_importance_raw_df.groupby(["dataset_id", "feature_set", "feature", "signal", "stat"], as_index=False)
    .agg(
        mean_coefficient=("coefficient", "mean"),
        mean_abs_coefficient=("abs_coefficient", "mean"),
        std_coefficient=("coefficient", "std"),
    )
    .sort_values(["dataset_id", "feature_set", "mean_abs_coefficient"], ascending=[True, True, False])
)
feature_importance_df.to_csv(FEATURE_IMPORTANCE_PATH, index=False, encoding="utf-8-sig")

cv_metrics_df.groupby(["dataset_id", "feature_set", "model"])[
    ["balanced_accuracy", "precision", "recall", "f1", "roc_auc", "fp", "fn"]
].mean().round(4)

balanced_accuracy  \
dataset_id                   feature_set model                                    
strict_no_event20            base70      dummy_most_frequent             0.5000   
                                         logistic_balanced               0.5738   
                             expanded154 dummy_most_frequent             0.5000   
                                         logistic_balanced               0.6220   
strict_no_event20_no_event67 base70      dummy_most_frequent             0.5000   
                                         logistic_balanced               0.5821   
                             expanded154 dummy_most_frequent             0.5000   
                                         logistic_balanced               0.6810   

                                                              precision  \
dataset_id                   feature_set model                            
strict_no_event20            base70      dummy_most_frequent     0.0000   
                                         logistic_balanced       0.3000   
                             expanded154 dummy_most_frequent     0.0000   
                                         logistic_balanced       0.3600   
strict_no_event20_no_event67 base70      dummy_most_frequent     0.0000   
                                         logistic_balanced       0.3333   
                             expanded154 dummy_most_frequent     0.0000   
                                         logistic_balanced       0.5033   

                                                              recall      f1  \
dataset_id                   feature_set model                                 
strict_no_event20            base70      dummy_most_frequent  0.0000  0.0000   
                                         logistic_balanced    0.5000  0.3714   
                             expanded154 dummy_most_frequent  0.0000  0.0000   
                                         logistic_balanced    0.5667  0.4205   
strict_no_event20_no_event67 base70      dummy_most_frequent  0.0000  0.0000   
                                         logistic_balanced    0.5333  0.4024   
                             expanded154 dummy_most_frequent  0.0000  0.0000   
                                         logistic_balanced    0.5667  0.5100   

                                                              roc_auc   fp  \
dataset_id                   feature_set model                               
strict_no_event20            base70      dummy_most_frequent   0.5000  0.0   
                                         logistic_balanced     0.6024  2.4   
                             expanded154 dummy_most_frequent   0.5000  0.0   
                                         logistic_balanced     0.7123  2.2   
strict_no_event20_no_event67 base70      dummy_most_frequent   0.5000  0.0   
                                         logistic_balanced     0.5813  2.6   
                             expanded154 dummy_most_frequent   0.5000  0.0   
                                         logistic_balanced     0.7135  1.4   

                                                               fn  
dataset_id                   feature_set model                     
strict_no_event20            base70      dummy_most_frequent  2.8  
                                         logistic_balanced    1.4  
                             expanded154 dummy_most_frequent  2.8  
                                         logistic_balanced    1.2  
strict_no_event20_no_event67 base70      dummy_most_frequent  2.6  
                                         logistic_balanced    1.2  
                             expanded154 dummy_most_frequent  2.6  
                                         logistic_balanced    1.0

## Threshold Results

In [7]:
threshold_rows = []
for dataset_id in DATASETS:
    for feature_set in FEATURE_SET_COLUMNS:
        prediction_part = cv_predictions_df.loc[
            cv_predictions_df["dataset_id"].eq(dataset_id)
            & cv_predictions_df["feature_set"].eq(feature_set)
            & cv_predictions_df["model"].eq("logistic_balanced")
        ].copy()
        y_true = prediction_part["actual_binary"].astype(int).to_numpy()
        y_score = prediction_part["positive_score"].astype(float).to_numpy()
        for threshold in THRESHOLDS:
            y_pred = (y_score >= threshold).astype(int)
            row = metric_row(y_true, y_pred, y_score)
            row.update(
                {
                    "dataset_id": dataset_id,
                    "feature_set": feature_set,
                    "model": "logistic_balanced",
                    "threshold": threshold,
                    "rows": int(len(prediction_part)),
                    "normal_rows": int((y_true == 0).sum()),
                    "positive_rows": int((y_true == 1).sum()),
                    "false_positive_rate": float(row["fp"] / (row["fp"] + row["tn"]))
                    if (row["fp"] + row["tn"])
                    else 0.0,
                    "false_negative_rate": float(row["fn"] / (row["fn"] + row["tp"]))
                    if (row["fn"] + row["tp"])
                    else 0.0,
                }
            )
            threshold_rows.append(row)

threshold_metrics_df = pd.DataFrame(threshold_rows)
threshold_metrics_df.to_csv(THRESHOLD_METRICS_PATH, index=False, encoding="utf-8-sig")
threshold_metrics_df.sort_values(["dataset_id", "feature_set", "threshold"]).round(4)

,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,tn,fp,fn,tp,dataset_id,feature_set,model,threshold,rows,normal_rows,positive_rows,false_positive_rate,false_negative_rate
0,0.5510,0.5357,0.3182,0.5000,0.3889,0.6041,20,15,7,7,strict_no_event20,base70,logistic_balanced,0.4,49,35,14,0.4286,0.5000
1,0.6122,0.5786,0.3684,0.5000,0.4242,0.6041,23,12,7,7,strict_no_event20,base70,logistic_balanced,0.5,49,35,14,0.3429,0.5000
2,0.6735,0.6214,0.4375,0.5000,0.4667,0.6041,26,9,7,7,strict_no_event20,base70,logistic_balanced,0.6,49,35,14,0.2571,0.5000
3,0.6735,0.6000,0.4286,0.4286,0.4286,0.6041,27,8,8,6,strict_no_event20,base70,logistic_balanced,0.7,49,35,14,0.2286,0.5714
4,0.6122,0.6000,0.3810,0.5714,0.4571,0.6776,22,13,6,8,strict_no_event20,expanded154,logistic_balanced,0.4,49,35,14,0.3714,0.4286
5,0.6531,0.6286,0.4211,0.5714,0.4848,0.6776,24,11,6,8,strict_no_event20,expanded154,logistic_balanced,0.5,49,35,14,0.3143,0.4286
6,0.7143,0.6500,0.5000,0.5000,0.5000,0.6776,28,7,7,7,strict_no_event20,expanded154,logistic_balanced,0.6,49,35,14,0.2000,0.5000
7,0.7143,0.6286,0.5000,0.4286,0.4615,0.6776,29,6,8,6,strict_no_event20,expanded154,logistic_balanced,0.7,49,35,14,0.1714,0.5714
8,0.6042,0.5835,0.3500,0.5385,0.4242,0.6484,22,13,6,7,strict_no_event20_no_event67,base70,logistic_balanced,0.4,48,35,13,0.3714,0.4615
9,0.6042,0.5835,0.3500,0.5385,0.4242,0.6484,22,13,6,7,strict_no_event20_no_event67,base70,logistic_balanced,0.5,48,35,13,0.3714,0.4615


## Results & Decision

In [8]:
def lookup_threshold(dataset_id: str, feature_set: str, metric: str) -> float:
    row = threshold_metrics_df.loc[
        threshold_metrics_df["dataset_id"].eq(dataset_id)
        & threshold_metrics_df["feature_set"].eq(feature_set)
        & threshold_metrics_df["threshold"].eq(DECISION_THRESHOLD)
    ].iloc[0]
    return float(row[metric])


decision_rows = []
for dataset_id in DATASETS:
    base = threshold_metrics_df.loc[
        threshold_metrics_df["dataset_id"].eq(dataset_id)
        & threshold_metrics_df["feature_set"].eq("base70")
        & threshold_metrics_df["threshold"].eq(DECISION_THRESHOLD)
    ].iloc[0]
    expanded = threshold_metrics_df.loc[
        threshold_metrics_df["dataset_id"].eq(dataset_id)
        & threshold_metrics_df["feature_set"].eq("expanded154")
        & threshold_metrics_df["threshold"].eq(DECISION_THRESHOLD)
    ].iloc[0]
    decision_rows.append(
        {
            "dataset_id": dataset_id,
            "threshold": DECISION_THRESHOLD,
            "base_balanced_accuracy": float(base["balanced_accuracy"]),
            "expanded_balanced_accuracy": float(expanded["balanced_accuracy"]),
            "balanced_accuracy_delta": float(expanded["balanced_accuracy"] - base["balanced_accuracy"]),
            "base_recall": float(base["recall"]),
            "expanded_recall": float(expanded["recall"]),
            "recall_delta": float(expanded["recall"] - base["recall"]),
            "base_false_positive_rate": float(base["false_positive_rate"]),
            "expanded_false_positive_rate": float(expanded["false_positive_rate"]),
            "false_positive_rate_delta": float(expanded["false_positive_rate"] - base["false_positive_rate"]),
            "base_f1": float(base["f1"]),
            "expanded_f1": float(expanded["f1"]),
            "f1_delta": float(expanded["f1"] - base["f1"]),
            "base_fp": int(base["fp"]),
            "expanded_fp": int(expanded["fp"]),
            "base_fn": int(base["fn"]),
            "expanded_fn": int(expanded["fn"]),
        }
    )

decision_matrix_df = pd.DataFrame(decision_rows)

main = decision_matrix_df.loc[decision_matrix_df["dataset_id"].eq("strict_no_event20")].iloc[0]
sensitivity = decision_matrix_df.loc[
    decision_matrix_df["dataset_id"].eq("strict_no_event20_no_event67")
].iloc[0]

passes_main_bacc = main["balanced_accuracy_delta"] >= 0.03
passes_main_recall = main["recall_delta"] >= -0.05
passes_main_fpr = main["false_positive_rate_delta"] <= 0.05
passes_sensitivity = sensitivity["balanced_accuracy_delta"] >= -0.03
feature_expansion_adopted = bool(
    passes_main_bacc and passes_main_recall and passes_main_fpr and passes_sensitivity
)

decision_matrix_df["passes_main_bacc"] = decision_matrix_df["dataset_id"].eq("strict_no_event20") & passes_main_bacc
decision_matrix_df["passes_main_recall"] = decision_matrix_df["dataset_id"].eq("strict_no_event20") & passes_main_recall
decision_matrix_df["passes_main_fpr"] = decision_matrix_df["dataset_id"].eq("strict_no_event20") & passes_main_fpr
decision_matrix_df["passes_sensitivity_bacc"] = decision_matrix_df["dataset_id"].eq("strict_no_event20_no_event67") & passes_sensitivity
decision_matrix_df["feature_expansion_adopted"] = feature_expansion_adopted
decision_matrix_df.to_csv(DECISION_MATRIX_PATH, index=False, encoding="utf-8-sig")

decision_text = (
    "expanded154를 다음 학습 기준 후보로 채택한다."
    if feature_expansion_adopted
    else "feature 확장만으로는 부족, normal negative 재설계 또는 센서 컬럼 확장 필요."
)

decision_matrix_df

,dataset_id,threshold,base_balanced_accuracy,expanded_balanced_accuracy,balanced_accuracy_delta,base_recall,expanded_recall,recall_delta,base_false_positive_rate,expanded_false_positive_rate,...,f1_delta,base_fp,expanded_fp,base_fn,expanded_fn,passes_main_bacc,passes_main_recall,passes_main_fpr,passes_sensitivity_bacc,feature_expansion_adopted
0,strict_no_event20,0.6,0.621429,0.650000,0.028571,0.500000,0.500000,0.000000,0.257143,0.200000,...,0.033333,9,7,7,7,False,True,True,False,False
1,strict_no_event20_no_event67,0.6,0.612088,0.659341,0.047253,0.538462,0.461538,-0.076923,0.314286,0.142857,...,0.048387,11,5,6,7,False,False,False,True,False


In [9]:
def markdown_table(df: pd.DataFrame, float_cols: list[str]) -> str:
    table = df.copy()
    for column in float_cols:
        if column in table.columns:
            table[column] = table[column].map(lambda value: f"{value:.4f}")
    columns = list(table.columns)
    lines = [
        "| " + " | ".join(str(column) for column in columns) + " |",
        "| " + " | ".join("---" for _ in columns) + " |",
    ]
    for _, row in table.iterrows():
        values = []
        for column in columns:
            value = row[column]
            values.append("" if pd.isna(value) else str(value).replace("|", "\\|"))
        lines.append("| " + " | ".join(values) + " |")
    return "\n".join(lines)


cv_summary = (
    cv_metrics_df.groupby(["dataset_id", "feature_set", "model"], as_index=False)
    .agg(
        balanced_accuracy=("balanced_accuracy", "mean"),
        precision=("precision", "mean"),
        recall=("recall", "mean"),
        f1=("f1", "mean"),
        roc_auc=("roc_auc", "mean"),
        fp=("fp", "mean"),
        fn=("fn", "mean"),
    )
    .sort_values(["dataset_id", "feature_set", "model"])
)

top_features = (
    feature_importance_df.loc[feature_importance_df["feature_set"].eq("expanded154")]
    .sort_values(["dataset_id", "mean_abs_coefficient"], ascending=[True, False])
    .groupby("dataset_id")
    .head(10)
    .reset_index(drop=True)
)

threshold_focus = threshold_metrics_df.sort_values(["dataset_id", "feature_set", "threshold"])
float_cols = [
    "accuracy",
    "balanced_accuracy",
    "precision",
    "recall",
    "f1",
    "roc_auc",
    "false_positive_rate",
    "false_negative_rate",
]

report = f'''
# M1 pre-event feature 확장 threshold 검증 보고서

## 결론

- 질문: 라벨은 그대로 잠그고 threshold 0.6 기준에서 feature 확장만 했을 때 성능이 좋아지는가?
- 결론: {decision_text}
- 개선은 있었다. 다만 main dataset의 balanced accuracy 개선폭이 사전 채택 기준 `+0.0300`에 비해 {0.03 - main['balanced_accuracy_delta']:+.4f}만큼 부족해 자동 채택으로 보지는 않았다.
- main dataset `strict_no_event20`의 threshold 0.6 balanced accuracy 변화는 {main['balanced_accuracy_delta']:+.4f}, recall 변화는 {main['recall_delta']:+.4f}, false positive rate 변화는 {main['false_positive_rate_delta']:+.4f}이다.
- Event 67 제외 민감도 dataset의 threshold 0.6 balanced accuracy 변화는 {sensitivity['balanced_accuracy_delta']:+.4f}이다.
- 이번 실험은 모델 고도화가 아니라 feature 확장 효과 검증이며, `normal vs fault_event` 전환은 보류했다.

## 데이터 구성

| 항목 | 값 |
|---|---:|
| accepted normal rows | {int(feature_df['label'].eq('normal').sum())} |
| accepted positive rows | {int(feature_df['label'].eq(POSITIVE_LABEL).sum())} |
| audit positive rows | {len(audit_df)} |
| base feature count | {len(BASE_FEATURE_COLUMNS)} |
| expanded feature count | {len(EXPANDED_FEATURE_COLUMNS)} |
| decision threshold | {DECISION_THRESHOLD} |

## Dataset Summary

{markdown_table(dataset_summary_df, ['coverage_min', 'coverage_median'])}

## Threshold 0.6 Decision Matrix

{markdown_table(decision_matrix_df, ['base_balanced_accuracy', 'expanded_balanced_accuracy', 'balanced_accuracy_delta', 'base_recall', 'expanded_recall', 'recall_delta', 'base_false_positive_rate', 'expanded_false_positive_rate', 'false_positive_rate_delta', 'base_f1', 'expanded_f1', 'f1_delta'])}

## Group CV 평균 성능

{markdown_table(cv_summary, ['balanced_accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'fp', 'fn'])}

## Threshold 0.4~0.7 성능

{markdown_table(threshold_focus, float_cols)}

## Expanded Feature 중요도 상위 후보

{markdown_table(top_features, ['mean_coefficient', 'mean_abs_coefficient', 'std_coefficient'])}

## 해석

- `base70`과 `expanded154`는 같은 label/window/dataset split에서 비교했다.
- 학습 입력에서는 metadata, 날짜, event id, `substation_id`, `coverage_rate`, `sample_count`, label strength를 제외했다.
- Event 20, 34, 69는 어떤 학습 dataset에도 포함하지 않았다.
- threshold 0.6은 최종 운영값이 아니라 기존 baseline에서 recall 유지와 false positive 감소가 동시에 나온 비교 기준이다.

## 생성 산출물

- `06_노트북/08_m1_pre_event_feature_expansion_threshold_validation.ipynb`
- `07_데이터산출물/m1_feature_expansion_audit.csv`
- `07_데이터산출물/m1_feature_expansion_features.csv`
- `07_데이터산출물/m1_feature_expansion_dataset_summary.csv`
- `07_데이터산출물/m1_feature_expansion_cv_metrics.csv`
- `07_데이터산출물/m1_feature_expansion_cv_predictions.csv`
- `07_데이터산출물/m1_feature_expansion_threshold_metrics.csv`
- `07_데이터산출물/m1_feature_expansion_feature_importance.csv`
- `07_데이터산출물/m1_feature_expansion_decision_matrix.csv`
'''.strip()

REPORT_PATH.write_text(report, encoding="utf-8")
print(decision_text)

feature 확장만으로는 부족, normal negative 재설계 또는 센서 컬럼 확장 필요.


## Checks

In [10]:
assert len(BASE_FEATURE_COLUMNS) == 70
assert len(EXPANDED_FEATURE_COLUMNS) == 154
assert len(set(EXPANDED_FEATURE_COLUMNS)) == 154
assert set(BASE_FEATURE_COLUMNS).issubset(EXPANDED_FEATURE_COLUMNS)

assert len(DATASETS["strict_no_event20"]) == 49
assert int(DATASETS["strict_no_event20"]["label"].eq("normal").sum()) == 35
assert int(DATASETS["strict_no_event20"]["label"].eq(POSITIVE_LABEL).sum()) == 14

assert len(DATASETS["strict_no_event20_no_event67"]) == 48
assert int(DATASETS["strict_no_event20_no_event67"]["label"].eq("normal").sum()) == 35
assert int(DATASETS["strict_no_event20_no_event67"]["label"].eq(POSITIVE_LABEL).sum()) == 13

for dataset_id, dataset in DATASETS.items():
    positive_events = set(dataset.loc[dataset["label"].eq(POSITIVE_LABEL), "source_event_id"].astype(int))
    assert EVENT20_ID not in positive_events
    assert EVENT34_ID not in positive_events
    assert EVENT69_ID not in positive_events

assert EVENT67_ID in set(
    DATASETS["strict_no_event20"].loc[
        DATASETS["strict_no_event20"]["label"].eq(POSITIVE_LABEL), "source_event_id"
    ].astype(int)
)
assert EVENT67_ID not in set(
    DATASETS["strict_no_event20_no_event67"].loc[
        DATASETS["strict_no_event20_no_event67"]["label"].eq(POSITIVE_LABEL), "source_event_id"
    ].astype(int)
)

assert len(cv_metrics_df) == 2 * 2 * 2 * 5
assert len(threshold_metrics_df) == 2 * 2 * len(THRESHOLDS)
assert len(decision_matrix_df) == 2
assert set(decision_matrix_df["threshold"]) == {DECISION_THRESHOLD}
assert len(feature_importance_df) == 2 * (len(BASE_FEATURE_COLUMNS) + len(EXPANDED_FEATURE_COLUMNS))

for _, row in cv_metrics_df.iterrows():
    train_groups = set(str(row["train_groups"]).split("|"))
    test_groups = set(str(row["test_groups"]).split("|"))
    assert train_groups.isdisjoint(test_groups)

metadata_like = {
    "sample_id",
    "manufacturer",
    "label",
    "label_strength",
    "source_event_id",
    "substation_id",
    "window_start",
    "window_end",
    "window_days",
    "window_policy",
    "source_type",
    "report_date",
    "possible_anomaly_start",
    "possible_anomaly_end",
    "training_start",
    "training_end",
    "fault_label",
    "fault_label_unknown",
    "efd_possible",
    "event67_flag",
    "audit_status",
    "exclusion_reason",
    "sample_count",
    "expected_sample_count",
    "coverage_rate",
    "disturbance_count",
    "disturbance_types",
    "dataset_id",
}
feature_like_columns = [column for column in feature_df.columns if column not in metadata_like]
assert sorted(feature_like_columns) == sorted(EXPANDED_FEATURE_COLUMNS)

blocked_terms = ["manufacturer" + "_2", "manufacturer" + " " + "2", "M" + "2"]
for output_path in [
    AUDIT_PATH,
    FEATURES_PATH,
    DATASET_SUMMARY_PATH,
    CV_METRICS_PATH,
    CV_PREDICTIONS_PATH,
    THRESHOLD_METRICS_PATH,
    FEATURE_IMPORTANCE_PATH,
    DECISION_MATRIX_PATH,
    REPORT_PATH,
]:
    text = output_path.read_text(encoding="utf-8")
    assert not any(term in text for term in blocked_terms), output_path

print("validation passed")
for output_path in [
    AUDIT_PATH,
    FEATURES_PATH,
    DATASET_SUMMARY_PATH,
    CV_METRICS_PATH,
    CV_PREDICTIONS_PATH,
    THRESHOLD_METRICS_PATH,
    FEATURE_IMPORTANCE_PATH,
    DECISION_MATRIX_PATH,
    REPORT_PATH,
]:
    print(output_path.relative_to(ROOT))

validation passed
07_데이터산출물\m1_feature_expansion_audit.csv
07_데이터산출물\m1_feature_expansion_features.csv
07_데이터산출물\m1_feature_expansion_dataset_summary.csv
07_데이터산출물\m1_feature_expansion_cv_metrics.csv
07_데이터산출물\m1_feature_expansion_cv_predictions.csv
07_데이터산출물\m1_feature_expansion_threshold_metrics.csv
07_데이터산출물\m1_feature_expansion_feature_importance.csv
07_데이터산출물\m1_feature_expansion_decision_matrix.csv
07_데이터산출물\08_M1_pre_event_feature_확장_threshold_검증_보고서.md
